### Porject is created based on https://www.youtube.com/watch?v=uz3LybMH7II

In [0]:
# upskill.pyspark_learning.menu_csv
# upskill.pyspark_learning.sales_csv

from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DateType

sales_schema = StructType([
    StructField("product_id",IntegerType(),True),
    StructField("customer_id",StringType(),True),
    StructField("order_date",DateType(),True),
    StructField("location",StringType(),True),
    StructField("source_order",StringType(),True)
])

In [0]:
#create sales data frame
sales_df = spark.read.csv('/Volumes/upskill/pyspark_learning/datasets/sales.csv.txt',schema =sales_schema)
display(sales_df)

product_id,customer_id,order_date,location,source_order
1,A,2023-01-01,India,Swiggy
2,A,2022-01-01,India,Swiggy
2,A,2023-01-07,India,Swiggy
3,A,2023-01-10,India,Restaurant
3,A,2022-01-11,India,Swiggy
3,A,2023-01-11,India,Restaurant
2,B,2022-02-01,India,Swiggy
2,B,2023-01-02,India,Swiggy
1,B,2023-01-04,India,Restaurant
1,B,2023-02-11,India,Swiggy


In [0]:
#Derive year,month, quarter
from pyspark.sql.functions import month,year,quarter
sales_df = sales_df.withColumn('order_year',year(sales_df.order_date))
sales_df = sales_df.withColumn('order_month',month(sales_df.order_date))
sales_df = sales_df.withColumn('order_quarter',quarter(sales_df.order_date))
display(sales_df)


product_id,customer_id,order_date,location,source_order,order_year,order_month,order_quarter
1,A,2023-01-01,India,Swiggy,2023,1,1
2,A,2022-01-01,India,Swiggy,2022,1,1
2,A,2023-01-07,India,Swiggy,2023,1,1
3,A,2023-01-10,India,Restaurant,2023,1,1
3,A,2022-01-11,India,Swiggy,2022,1,1
3,A,2023-01-11,India,Restaurant,2023,1,1
2,B,2022-02-01,India,Swiggy,2022,2,1
2,B,2023-01-02,India,Swiggy,2023,1,1
1,B,2023-01-04,India,Restaurant,2023,1,1
1,B,2023-02-11,India,Swiggy,2023,2,1


In [0]:
# create menu data frame

from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DateType

menu_schema = StructType([
    StructField("product_id",IntegerType(),True),
    StructField("product_name",StringType(),True),
    StructField("price",IntegerType(),True)
])

menu_df = spark.read.csv('/Volumes/upskill/pyspark_learning/datasets/menu.csv.txt',schema =menu_schema)
display(menu_df)

product_id,product_name,price
1,PIZZA,100
2,Chowmin,150
3,sandwich,120
4,Dosa,110
5,Biryani,80
6,Pasta,180


In [0]:
# Total Amount spent by each customer
total_amount_spent = (sales_df.join(menu_df,'product_id').groupBy('customer_id').agg({'price':'sum'}).orderBy('customer_id'))
display(total_amount_spent)

customer_id,sum(price)
A,4260
B,4440
C,2400
D,1200
E,2040


Databricks visualization. Run in Databricks to view.

In [0]:
total_amount_spent_cat = sales_df.join(menu_df,'product_id').groupBy('product_name').agg({'price':'sum'}).orderBy('product_name')
display(total_amount_spent_cat)

product_name,sum(price)
Biryani,480
Chowmin,3600
Dosa,1320
PIZZA,2100
Pasta,1080
sandwich,5760


Databricks visualization. Run in Databricks to view.

In [0]:
# Total amount spend by each category
total_amount_spent_cat = (sales_df.join(menu_df,'product_id').groupBy('product_name').agg({'price':'sum'}).orderBy('product_name'))
display(total_amount_spent_cat)

product_name,sum(price)
Biryani,480
Chowmin,3600
Dosa,1320
PIZZA,2100
Pasta,1080
sandwich,5760


In [0]:
# yearly sale
yearly_sales = (sales_df.join(menu_df,'product_id').groupBy('order_year').agg({'price':'sum'}).orderBy('order_year'))
display(yearly_sales)

order_year,sum(price)
2022,4350
2023,9990


Databricks visualization. Run in Databricks to view.

In [0]:
# Total sales in each month
total_amount_spent_month = (sales_df.join(menu_df,'product_id').groupBy('order_month').agg({'price':'sum'}).orderBy('order_month'))
display(total_amount_spent_month)


order_month,sum(price)
1,2960
2,2730
3,910
5,2960
6,2960
7,910
11,910


Databricks visualization. Run in Databricks to view.

In [0]:
# how many times each menu was ordered
from pyspark.sql.functions import count
menu_count = sales_df.join(menu_df,'product_id').groupBy('product_id','product_name').agg(count('product_id').alias('product_count')).orderBy('product_count',ascending=False).drop('product_id')
display(menu_count)

product_name,product_count
sandwich,48
Chowmin,24
PIZZA,21
Dosa,12
Biryani,6
Pasta,6


Databricks visualization. Run in Databricks to view.

In [0]:
# top 5 ordered items
from pyspark.sql.functions import count
top5_items = sales_df.join(menu_df,'product_id').groupBy('product_id','product_name').agg(count('product_id').alias('product_count')).orderBy('product_count',ascending=False).drop('product_id','product_count').limit(5)
display(top5_items)


product_name
sandwich
Chowmin
PIZZA
Dosa
Biryani


In [0]:
# frequency customer visiting restaurant
from pyspark.sql.functions import countDistinct
frequency_customer = sales_df.filter(sales_df.source_order == 'Restaurant').groupBy('customer_id').agg(countDistinct('order_date').alias('frequency')).orderBy('frequency',ascending=False)
display(frequency_customer)

customer_id,frequency
B,6
A,6
E,5
C,3
D,1


Databricks visualization. Run in Databricks to view.

In [0]:
# total sales for each country
from pyspark.sql.functions import sum
total_sales_country = (sales_df.join(menu_df,'product_id').groupBy('location').agg(sum("price").alias("total_sales")).orderBy('total_sales',ascending= False))
display(total_sales_country)

location,total_sales
UK,7020
India,4860
USA,2460


Databricks visualization. Run in Databricks to view.

In [0]:
# total sales by Order Source
from pyspark.sql.functions import sum
total_sales_order_source = (sales_df.join(menu_df,'product_id').groupBy('Source_Order').agg(sum("price").alias("total_sales")).orderBy('total_sales',ascending= False))
display(total_sales_order_source)

Source_Order,total_sales
Swiggy,6330
zomato,4920
Restaurant,3090


Databricks visualization. Run in Databricks to view.